# Predicting Breast Cancer Molecular Subtype from Gene Expression

**Virginia Galván, PhD** · Bioinformatics · Genomic & Multi-Omics Data Science

This notebook is the curated, end-to-end walkthrough of the project: from raw public gene expression data to a deployed prediction API. It pulls together the results of the three detailed working notebooks (`notebooks/01–03`) without re-running any analysis — every figure and number here was generated there and is loaded/referenced as-is.

**Live API:** [tcga-brca-subtype-classifier.onrender.com/docs](https://tcga-brca-subtype-classifier.onrender.com/docs)

## Table of Contents

1. [The Problem](#1-the-problem)
2. [Data & Cohort](#2-data--cohort)
3. [Exploratory Analysis](#3-exploratory-analysis)
4. [Model Comparison](#4-model-comparison)
5. [Test Set Evaluation](#5-test-set-evaluation)
6. [Interpretability (SHAP)](#6-interpretability-shap)
7. [From Notebook to Production](#7-from-notebook-to-production)
8. [Conclusions & Next Steps](#8-conclusions--next-steps)

## 1. The Problem

Breast cancer isn't one disease. Molecular subtype (PAM50: Luminal A, Luminal B, HER2-enriched, Basal-like, Normal-like) is what determines whether a patient receives hormone therapy, HER2-targeted therapy, or chemotherapy — so predicting it from gene expression is a real clinical decision-support task, not a synthetic benchmark.

This project frames subtype prediction as a supervised multi-class classification problem: three models are compared under identical cross-validation conditions, the best one is interpreted with SHAP and checked against known cancer biology, and it's deployed end-to-end as a live REST API — not left as a notebook result.

## 2. Data & Cohort

- **Source:** [cBioPortal for Cancer Genomics](https://www.cbioportal.org) public REST API — study `brca_tcga_pan_can_atlas_2018` (TCGA, Breast Invasive Carcinoma, PanCancer Atlas). No bulk file download; clinical and expression data are queried directly.
- **Features:** the PAM50 50-gene panel (Parker et al. 2009, *J Clin Oncol*) — a clinically-validated gene set already established as optimally discriminative for these subtypes, used here as the feature selection step rather than re-derived from the full ~20,000-gene transcriptome.
- **Cohort:** 981 patients with complete PAM50 expression and a subtype label. Classes are imbalanced, consistent with population-level PAM50 distributions:

| Subtype | n |
|---|---|
| Luminal A | 499 |
| Luminal B | 197 |
| Basal-like | 171 |
| HER2-enriched | 78 |
| Normal-like | 36 |

- Clean data is loaded into SQL (SQLite, portable to Postgres) and the modeling cohort is assembled with a SQL query — see `notebooks/01_data_acquisition_qc.ipynb`.

![Class distribution](figures/fig1_subtype_class_distribution.png)

## 3. Exploratory Analysis

Before modeling, the PAM50 panel is checked for structure and for consistency with known subtype biology.

**PCA** shows samples forming partially overlapping clusters that align with subtype along the first two components — expected, since PAM50 is defined directly from expression patterns in this gene panel, and adjacent subtypes (e.g. Luminal A/B) represent a graded hormone-receptor and proliferation axis rather than fully discrete categories.

![PCA by subtype](figures/fig2_pca_by_subtype.png)

**Gene-gene correlation** reveals two recognized co-expression modules: a proliferation block (MKI67, BIRC5, CCNB1, RRM2, …) and a hormone-receptor block (ESR1, PGR, FOXA1, …). This matters later — it's the reason a single proliferation marker like MKI67 doesn't need to rank high individually for the model to use proliferation signal (Section 6).

![Gene correlation heatmap](figures/fig3_gene_correlation_heatmap.png)

**Marker genes** behave as expected biologically: ESR1 is highest in Luminal A/B, ERBB2 in HER2-enriched, MKI67 (proliferation) in Basal-like — a direct sanity check on the data before any modeling happens.

![Marker genes by subtype](figures/fig4_marker_genes_by_subtype.png)

## 4. Model Comparison

Logistic Regression, Random Forest, and XGBoost are compared with 5-fold stratified cross-validation on the training set (80% of the cohort). Class imbalance is handled identically for all three — per-sample weights at fit time — so the comparison isn't confounded by each model handling imbalance differently.

| Model | Accuracy | F1-macro | ROC-AUC (OvR) |
|---|---|---|---|
| Logistic Regression | 0.83 | 0.78 | 0.96 |
| Random Forest | 0.90 | 0.81 | 0.99 |
| **XGBoost (best)** | **0.91** | **0.86** | **0.99** |

XGBoost wins on F1-macro — the metric that matters most here, since it weights all 5 subtypes equally rather than being dominated by the majority class (Luminal A). It's refit on the full training set and taken forward as the final model.

![Model comparison](figures/fig5_model_comparison_cv.png)

## 5. Test Set Evaluation

The final XGBoost model is evaluated once on the held-out test set (20%, never seen during model selection): **93% accuracy**, F1-macro 0.93.

| Subtype | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Basal-like | 1.00 | 1.00 | 1.00 | 34 |
| HER2-enriched | 0.88 | 0.94 | 0.91 | 16 |
| Luminal A | 0.93 | 0.94 | 0.94 | 100 |
| Luminal B | 0.87 | 0.85 | 0.86 | 40 |
| Normal-like | 1.00 | 0.86 | 0.92 | 7 |

Basal-like — the most clinically distinct subtype — is predicted perfectly. Luminal A/B, which overlap biologically (Section 3), are where most of the model's confusion happens, which is the expected failure mode rather than a red flag. Normal-like has only 7 test samples, so its recall (0.86 → 6/7 correct) should be read with that small sample size in mind rather than as a precise estimate.

![Confusion matrix](figures/fig6_confusion_matrix_test.png)

## 6. Interpretability (SHAP)

SHAP (TreeExplainer) values quantify which genes drive each prediction, and let the model's learned reasoning be checked against established breast-cancer biology — without ever telling the model anything about cancer.

![SHAP global importance](figures/fig7_shap_global_importance.png)

**Biological cross-validation**, checking the three marker genes examined in Section 3 against their SHAP ranking out of all 50 PAM50 genes:

| Gene | SHAP rank | Known role |
|---|---|---|
| ESR1 | **1 / 50** | Estrogen receptor — the gene oncologists use to decide hormone therapy |
| ERBB2 | **7 / 50** | HER2 — target of trastuzumab |
| MKI67 | 47 / 50 | Proliferation marker |

ESR1 landing at rank 1 with no prior biological input is the headline result. MKI67 ranking low individually is *not* a failure — Section 3's correlation heatmap shows proliferation genes (MKI67, BIRC5, CCNB1, RRM2, …) are tightly co-expressed, so the model can capture proliferation signal by leaning on any of several correlated genes, diluting the SHAP credit any single one gets. Reading that nuance correctly, rather than only reporting the genes that rank neatly at the top, is part of validating a model honestly.

![Single prediction explanation](figures/fig8_shap_single_prediction.png)

## 7. From Notebook to Production

The trained model isn't left as a notebook artifact:

1. The fitted model, PAM50 gene order, and label encoder are exported together as one bundle (`api/model.joblib`), so predictions can be reproduced exactly outside the notebook.
2. A **FastAPI** app (`api/main.py`) serves it behind a `/predict` endpoint (and a CSV batch endpoint — see `examples/sample_patients.csv`).
3. The app is **Dockerized** and deployed to **Render**, with a GitHub Actions workflow keeping the free-tier instance warm.

**Try it live:** [tcga-brca-subtype-classifier.onrender.com/docs](https://tcga-brca-subtype-classifier.onrender.com/docs) (interactive Swagger UI — first request after idle time can take ~50s while the free tier wakes up).

## 8. Conclusions & Next Steps

This project takes a real clinical prediction task — breast cancer molecular subtyping — through the full pipeline a production model needs: public data acquisition via API, SQL-based cohort assembly, model comparison under matched conditions, held-out evaluation, biological validation of what the model actually learned, and deployment as a live, queryable service.

The result that matters most isn't the 93% accuracy alone — it's that the model's own reasoning, recovered independently through SHAP, converges on the same genes (ESR1, ERBB2) that oncologists already use to make treatment decisions.

**Next steps:** extend beyond the PAM50 panel to test whether additional genes improve separation of the hardest cases (Luminal A/B); add monitoring for prediction drift if the API sees real traffic; expand test coverage for the API's CSV batch endpoint.

**Full technical detail:** `notebooks/01_data_acquisition_qc.ipynb` → `02_modeling_cross_validation.ipynb` → `03_shap_interpretability.ipynb`.

---

**Virginia Galván, PhD** · Bioinformatics · Genomic & Multi-Omics Data Science · [LinkedIn](https://www.linkedin.com/in/virgina-galvan-390ba233b/) · [GitHub](https://github.com/virginiagalvan)